![image.png](https://i.imgur.com/4fN73lZ.png)

# Continuous Control with PPO: MountainCarContinuous

> **Optional, solved lab** for Part II of the lecture. Run it top to bottom: training takes under a minute on a laptop CPU.

A car sits in a valley and has to reach the flag on the hill to its right, but its engine is too weak to drive straight up. It has to rock back and forth to build momentum. The action is a **continuous** force between $-1$ and $1$, so there is no table of actions for Q-learning's $\max_a Q(s,a)$ to search. A **policy gradient** method does not need one: the policy outputs a *distribution* over forces, and learning makes the better forces more likely. We use **PPO**, with the same clipped objective as the lecture.

What you will see:

1. **Hand-written policies**, as in Lab 2. Doing nothing scores 0, random pushing about $-50$, and "push the way you are moving" about 89.
2. **Plain PPO learns to do nothing.** Every push costs a little, and random pushing almost never finds the flag, so standing still looks best.
3. **Reward shaping** with the car's mechanical energy fixes the exploration problem *without changing which policy is best*. PPO then scores about **94**: above the official "solved" bar of 90, and above the hand-written rule.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control]" imageio matplotlib torch

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import gymnasium as gym

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

env = gym.make("MountainCarContinuous-v0")
print("observation:", env.observation_space)
print("action:     ", env.action_space)
print("time limit: ", env.spec.max_episode_steps, "steps")

## The Environment

| | |
|---|---|
| **State** | position $x \in [-1.2, 0.6]$ and velocity $v \in [-0.07, 0.07]$ |
| **Action** | engine force $a \in [-1, 1]$, continuous |
| **Reward** | $-0.1\,a^2$ every step, and $+100$ on reaching the flag at $x \ge 0.45$ |
| **Episode end** | reaching the flag, or 999 steps |

The reward makes this a hard **exploration** problem. Until the car reaches the flag for the first time, the only signal is the cost of pushing, and that signal says *push less*.

First, some hand-written policies, scored by their average return over 100 episodes:

In [ ]:
def evaluate(policy, n_episodes=100, seed=1000):
    # Average return, and average episode length, of a deterministic policy: state -> force.
    returns, lengths = [], []
    for i in range(n_episodes):
        state, _ = env.reset(seed=seed + i)
        done, total, steps = False, 0.0, 0
        while not done:
            force = np.array([policy(state)], dtype=np.float32)
            state, reward, terminated, truncated, _ = env.step(force)
            total += reward
            steps += 1
            done = terminated or truncated
        returns.append(total)
        lengths.append(steps)
    return np.mean(returns), np.mean(lengths)

rng = np.random.default_rng(SEED)
baselines = {
    "do nothing":                  lambda s: 0.0,
    "random force":                lambda s: float(np.clip(rng.normal(), -1, 1)),
    "push the way you are moving": lambda s: 1.0 if s[1] >= 0 else -1.0,
}
for name, policy in baselines.items():
    score, length = evaluate(policy)
    print(f"{name:<29} score {score:7.2f}   episode length {length:5.0f} steps")

Pushing at random costs about 50 and almost never reaches the flag. Pushing the way the car is already moving pumps energy in, like a swing, and reaches the flag in about 100 steps; full force all the way costs about 10 of the 100 points.

## A Gaussian Policy for Continuous Actions

With discrete actions the policy was a softmax over a few choices. With a continuous action it is a **Gaussian**: the network outputs a mean force $\mu_\theta(s)$, a learned standard deviation $\sigma$ sets how widely to explore, and

$$\pi_\theta(a \mid s) = \mathcal{N}\big(\mu_\theta(s),\ \sigma^2\big).$$

PPO only needs $\log \pi_\theta(a \mid s)$ of the sampled force to form its ratio $r_t(\theta)$, exactly as in the lecture. The force is clipped to $[-1, 1]$ before it reaches the environment. A critic $V_\phi(s)$ provides the baseline, and the advantages come from GAE.

In [ ]:
def normalize(state):
    # Scale position and velocity to roughly [-1, 1]; small networks learn faster on inputs that size.
    return np.array([(state[0] + 0.3) / 0.9, state[1] / 0.07], dtype=np.float32)

def mlp():
    return nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))

## Reward Shaping That Keeps the Best Policy

The car's mechanical energy is its height plus its speed:

$$E(s) = \tfrac{0.0025}{3}\,\sin(3x) + \tfrac{1}{2}\,v^2,$$

where $0.0025$ is this environment's gravity. Rocking the car raises its energy long before it reaches the flag, so rewarding energy gains gives the learner a signal from the very first episodes.

We add that signal as a **potential-based** shaping term, with potential $\Phi(s) = K\,E(s)$:

$$r'_t = r_t + \gamma\,\Phi(s_{t+1}) - \Phi(s_t).$$

Summed over an episode, the shaping terms telescope: their discounted sum depends only on where the episode starts and ends, not on the path in between. So they cannot change which policy is best; they only change how quickly the learner finds it (Ng, Harada and Russell, 1999). An episode that reaches the flag ends with potential 0. The shaped reward is used for **training only**: every score in this notebook is the environment's own reward.

In [ ]:
def energy(state):
    position, velocity = state
    return 0.0025 / 3 * np.sin(3 * position) + 0.5 * velocity ** 2

def shaped_reward(reward, state, next_state, terminated, K, gamma):
    phi = K * energy(state)
    phi_next = 0.0 if terminated else K * energy(next_state)
    return reward + gamma * phi_next - phi

## PPO

One update: run the current policy for 2,048 steps, compute advantages with GAE, then take a few epochs of minibatch gradient steps on the clipped objective

$$L^{\text{CLIP}}(\theta) = -\,\mathbb{E}_t\Big[\min\big(r_t(\theta)\,\hat{A}_t,\ \operatorname{clip}\big(r_t(\theta), 1-\varepsilon, 1+\varepsilon\big)\,\hat{A}_t\big)\Big],$$

plus the critic's squared error. These are the lecture's PPO slides, written out in full.

In [ ]:
def train_ppo(K, total_steps, seed=SEED, n_steps=2048, epochs=10, minibatch=64, lr=3e-4,
              gamma=0.99, lam=0.95, clip=0.2, log_every=10):
    # Train PPO with shaping strength K (K = 0 means no shaping). Returns the actor and the
    # environment's own return for every training episode.
    torch.manual_seed(seed)
    np.random.seed(seed)
    actor, critic = mlp(), mlp()
    log_std = nn.Parameter(torch.zeros(1))   # sigma starts at 1: wide exploration
    params = list(actor.parameters()) + list(critic.parameters()) + [log_std]
    optimizer = torch.optim.Adam(params, lr=lr)

    state, _ = env.reset(seed=seed)
    episode_return, episode_returns, steps, update, start = 0.0, [], 0, 0, time.time()
    while steps < total_steps:
        # 1. Collect a rollout with the current policy.
        states, actions, logps, rewards, terminals, values = [], [], [], [], [], []
        for _ in range(n_steps):
            x = torch.as_tensor(normalize(state))
            with torch.no_grad():
                dist = torch.distributions.Normal(actor(x), log_std.exp())
                action = dist.sample()
                value = critic(x).squeeze()
            next_state, reward, terminated, truncated, _ = env.step(action.clamp(-1, 1).numpy())
            episode_return += reward
            states.append(x); actions.append(action); logps.append(dist.log_prob(action).sum())
            rewards.append(shaped_reward(reward, state, next_state, terminated, K, gamma))
            terminals.append(terminated); values.append(value)
            state, steps = next_state, steps + 1
            if terminated or truncated:
                episode_returns.append(episode_return)
                episode_return = 0.0
                state, _ = env.reset()

        # 2. Advantages with GAE. A time-limit cut bootstraps from V(s'); reaching the flag does not.
        with torch.no_grad():
            values.append(critic(torch.as_tensor(normalize(state))).squeeze())
        advantages, gae = torch.zeros(n_steps), 0.0
        for t in reversed(range(n_steps)):
            not_done = 0.0 if terminals[t] else 1.0
            delta = rewards[t] + gamma * values[t + 1] * not_done - values[t]
            gae = delta + gamma * lam * not_done * gae
            advantages[t] = gae
        returns = advantages + torch.stack(values[:-1])
        states, actions, logps = torch.stack(states), torch.stack(actions), torch.stack(logps)

        # 3. A few epochs of PPO-Clip on minibatches.
        for _ in range(epochs):
            for batch in torch.randperm(n_steps).split(minibatch):
                dist = torch.distributions.Normal(actor(states[batch]), log_std.exp())
                ratio = (dist.log_prob(actions[batch]).sum(-1) - logps[batch]).exp()
                adv = advantages[batch]
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)
                policy_loss = -torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv).mean()
                value_loss = 0.5 * (critic(states[batch]).squeeze(-1) - returns[batch]).pow(2).mean()
                optimizer.zero_grad()
                (policy_loss + value_loss).backward()
                nn.utils.clip_grad_norm_(params, 0.5)
                optimizer.step()

        update += 1
        if update % log_every == 0:
            recent = np.mean(episode_returns[-10:]) if episode_returns else float("nan")
            print(f"step {steps:7d} | episodes {len(episode_returns):5d} | "
                  f"return of the last 10 (true reward) {recent:7.2f} | sigma {log_std.exp().item():.2f} | "
                  f"{time.time() - start:4.0f}s")
    return actor, episode_returns

def greedy(actor):
    # The trained policy without exploration noise: always use the mean force.
    def policy(state):
        with torch.no_grad():
            return float(actor(torch.as_tensor(normalize(state))).clamp(-1, 1))
    return policy

## Plain PPO: Learning to Do Nothing

First, PPO on the environment's own reward ($K = 0$):

In [ ]:
plain_actor, plain_returns = train_ppo(K=0.0, total_steps=60_000)
score, length = evaluate(greedy(plain_actor))
print(f"\nplain PPO: score {score:.2f}, episode length {length:.0f} steps")

It scores 0 and every episode runs out the clock: the policy learned to stop pushing. It never reached the flag, so it never saw the $+100$, and the only lesson left in the data was that pushing costs.

## PPO With Energy Shaping

In [ ]:
K = 10_000   # scales energy (about 0.002 from valley to flag) to about 20 reward points
actor, train_returns = train_ppo(K=K, total_steps=200_000)
score, length = evaluate(greedy(actor))
print(f"\nshaped PPO: score {score:.2f} over 100 episodes, episode length {length:.0f} steps")
print("solved (an average of at least 90):", score >= 90)

In [ ]:
window = 20
smooth = np.convolve(train_returns, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 3.5))
plt.plot(smooth, label=f"shaped PPO, true reward (moving average of {window})")
plt.axhline(90, color="green", linestyle="--", label="solved (90)")
plt.axhline(0, color="gray", linestyle=":", label="plain PPO / do nothing (0)")
plt.xlabel("training episode")
plt.ylabel("episode return")
plt.legend()
plt.show()

## What Did It Learn?

The left plot is the force the trained policy chooses at each position and velocity. The right one is the hand-written rule from the start of the notebook. They agree on most of the map, and differ in two places. When the car is nearly at rest, PPO pushes *left*, backing up the hill first. High on the left hill it pushes *right* while the car is still rolling left, turning the swing around early.

In [ ]:
positions = np.linspace(-1.2, 0.6, 121)
velocities = np.linspace(-0.07, 0.07, 121)
learned = np.array([[greedy(actor)((p, v)) for p in positions] for v in velocities])
hand = np.array([[1.0 if v >= 0 else -1.0 for p in positions] for v in velocities])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, grid, title in [(axes[0], learned, "learned by PPO"), (axes[1], hand, "push the way you are moving")]:
    im = ax.imshow(grid, origin="lower", aspect="auto", cmap="coolwarm", vmin=-1, vmax=1,
                   extent=[positions[0], positions[-1], velocities[0], velocities[-1]])
    ax.axvline(0.45, color="black", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("position (flag at 0.45)")
axes[0].set_ylabel("velocity")
fig.colorbar(im, ax=axes, label="force")
plt.show()

for name, policy in [("learned by PPO", greedy(actor)), ("push the way you are moving", baselines["push the way you are moving"])]:
    forces, done = [], False
    state, _ = env.reset(seed=1000)
    while not done:
        force = policy(state)
        forces.append(force)
        state, reward, terminated, truncated, _ = env.step(np.array([force], dtype=np.float32))
        done = terminated or truncated
    turns = sum(np.sign(a) != np.sign(b) for a, b in zip(forces, forces[1:]))
    print(f"{name:<28} first push {'left ' if forces[0] < 0 else 'right'} | {turns} change{'s' if turns != 1 else ''} of direction | {len(forces)} steps")

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image, display
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless rendering, e.g. on Colab
os.makedirs("video", exist_ok=True)

render_env = gym.make("MountainCarContinuous-v0", render_mode="rgb_array")
state, _ = render_env.reset(seed=1000)
frames, done = [render_env.render()], False
while not done:
    state, reward, terminated, truncated, _ = render_env.step(np.array([greedy(actor)(state)], dtype=np.float32))
    frames.append(render_env.render())
    done = terminated or truncated
render_env.close()
imageio.mimsave("video/mountain_car_ppo.gif", frames, fps=30, loop=0)
display(Image(filename="video/mountain_car_ppo.gif"))

## Takeaways

- **Continuous actions are where policy gradients shine.** A Gaussian policy needs only a log-probability for PPO's ratio, never a maximum over actions.
- **The hard part of MountainCar is exploration, not control.** With a reward that arrives only at the flag and a cost on every push, plain PPO settles for doing nothing.
- **Potential-based shaping adds a denser signal without changing which policy is best.** A shaping bonus that is *not* potential-based changes the goal itself, and the policy will optimise the bonus instead: the reward hacking of Part III.
- **The learned policy beats the obvious rule by planning a better swing.** From rest it first backs up the left hill, then drives right in one go. The rule pushes right first, rolls back and needs more swings. Both push at nearly full force; PPO simply needs fewer steps, so it pays for fewer pushes.

## Credits & Further Reading

- **PPO**: Schulman, Wolski, Dhariwal, Radford, Klimov, *Proximal Policy Optimization Algorithms*, 2017. [arxiv.org/abs/1707.06347](https://arxiv.org/abs/1707.06347)
- **GAE**: Schulman, Moritz, Levine, Jordan, Abbeel, *High-Dimensional Continuous Control Using Generalized Advantage Estimation*, ICLR 2016. [arxiv.org/abs/1506.02438](https://arxiv.org/abs/1506.02438)
- **Potential-based shaping**: Ng, Harada, Russell, *Policy Invariance Under Reward Transformations: Theory and Application to Reward Shaping*, ICML 1999.
- **Mountain Car**: Moore, *Efficient Memory-based Learning for Robot Control*, PhD thesis, University of Cambridge, 1990; the continuous version is [Gymnasium's `MountainCarContinuous-v0`](https://gymnasium.farama.org/environments/classic_control/mountain_car_continuous/).